In [1]:
%load_ext autoreload
%autoreload 2

from tasks.autoencoder import AETask
import os
import torch
from tqdm import tqdm
import einx
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import Pooling, Transformer, Normalize
from transformers import AutoModel, AutoTokenizer, T5TokenizerFast, AutoModelForCausalLM, T5Tokenizer
from datasets import load_dataset, load_from_disk
import re
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
from data.datasets import WikipediaDataset, WikipediaDatasetConfig
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, models
import wandb
from tasks.dlclm import DLCLMTask
from mauve import compute_mauve, get_features_from_input
from model.encoder import HSEMHead, HSEMHeadConfig, SEMHeadConfig, SEMHead
import os
from data.datasets import FineWebDataset

/home/l/leog/links/scratch/sentence_diffusion/venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/l/leog/links/scratch/sentence_diffusion/venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assign

In [38]:
data = FineWebDataset(length_interval=[32, 96])

Resolving data files:   0%|          | 0/824 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/828 [00:00<?, ?it/s]

In [44]:
batch = data.__getitems__([0,1,2,3,4,5,6,7,8,9,10,11,12])

In [207]:
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device='cuda')

# We recommend enabling flash_attention_2 for better acceleration and memory saving,
# together with setting `padding_side` to "left":
# model = SentenceTransformer(
#     "Qwen/Qwen3-Embedding-0.6B",
#     model_kwargs={"attn_implementation": "flash_attention_2", "device_map": "auto"},
#     tokenizer_kwargs={"padding_side": "left"},
# )

In [208]:
#prompt = "Represent the text with high-level features. This can include semantic information, type of text, intent, style. Text: "
#prompt = "Instruct: Represent the text with high-level features. This can include semantic information, type of text, intent, style.\nText: "
prompt = None
query_embeddings = model.encode(batch["input_str"], prompt=prompt)
document_embeddings = model.encode(batch["input_str"], prompt=prompt)
similarity = model.similarity(query_embeddings, document_embeddings)
similarity.fill_diagonal_(-torch.inf);

In [204]:
batch["input_str"][1]

"\nI'm not going to lie to you and say that I loved everything about it (3:00 AM wake up calls being the main offender), but I was consistently surprised to find that even in the tougher times, when we had been blearily working for 18 hours straight, something or someone would come along to pick everyone up.\nFrom the absolute chaos of pre-show preparations,"

In [206]:
batch["input_str"][8]

' making Monday nights on SPEED interesting again. It also helped that the network executives finally relented and let the race review lead the show. It made a big difference.\nThis week, Byrnes, Waltrip and Knaus were all tired from a long California weekend and a three hour time zone shift. Waltrip started slow, but got himself back on-track by suggesting California go to'

In [205]:
torch.softmax(similarity/0.01,-1)[1].argsort(descending=True)

tensor([ 8,  9,  7, 11,  6, 12, 10,  3,  2,  4,  5,  0,  1])

In [210]:
torch.softmax(similarity/0.01,-1)[0].argsort(descending=True)

tensor([ 7, 12,  2, 10,  1,  9, 11,  4,  3,  8,  6,  5,  0])

In [169]:
print("A:" +batch["input_str"][1] + "\nB:" + batch["input_str"][6])

A:
I'm not going to lie to you and say that I loved everything about it (3:00 AM wake up calls being the main offender), but I was consistently surprised to find that even in the tougher times, when we had been blearily working for 18 hours straight, something or someone would come along to pick everyone up.
From the absolute chaos of pre-show preparations,
B:ly got 28 first-place votes versus only 11 for Wagner.
At times like these I can imagine some fans crying about an East Coast bias within the media, but I'm not sure how much I buy that here given the fact that the


In [164]:
# The queries and documents to embed
queries = [
    "Then, I went to the garden to the garden to pick some cherries.",
    "Explain gravity.",
]
documents = [
    "Last, year, diego went to Africa to meet an old friend.",
    "Gravity is a force that attracts two bodies towards each other.",
    "I love going in the forest and foraging mushrooms!",
    "You love going in the forest and foraging mushrooms!",
    "Can you explain the maillard reaction?",
    "Explain why the sky is blue.",
    "Yesterday, I went to the store to buy some things."
]
prompt = None

#prompt = "Instruct: Given some text, retrieve other texts which share some high level attributes. This can include semantic information, type of text, intent, style. \nText:"
prompt = "Represent the text with high-level features. This can include semantic information, type of text, intent, style. Text: "
#prompt = "Instruct: Given some text taken from the internet, retrieve other similar text. Text: "
#prompt = "Instruct: Represent the following passage with its high-level features. This can include semantic information, type of text, intent, style.\nText: "
#prompt = "Instruct: Represent the following text "
query_embeddings = model.encode(queries, prompt=prompt)
document_embeddings = model.encode(documents, prompt=prompt)

In [165]:
similarity = model.similarity(query_embeddings, document_embeddings)
similarity

tensor([[0.8175, 0.8386, 0.9061, 0.8952, 0.8412, 0.8487, 0.8798],
        [0.8261, 0.9366, 0.9186, 0.9336, 0.9719, 0.9821, 0.8365]])

In [166]:
torch.softmax(similarity/0.1,-1)

tensor([[0.0882, 0.1089, 0.2141, 0.1919, 0.1119, 0.1206, 0.1645],
        [0.0509, 0.1537, 0.1284, 0.1492, 0.2189, 0.2424, 0.0565]])

In [153]:
import torch

In [15]:
x = torch.IntTensor([[1,2,3],[4,5,6]])
prompt = [3,3,4,4]

In [22]:
[[] for i in range(10)]

[[], [], [], [], [], [], [], [], [], []]

In [18]:
torch.cat([x,y])

RuntimeError: Tensors must have same number of dimensions: got 2 and 1

In [7]:
[prompt + x]

TypeError: can only concatenate list (not "Tensor") to list